# P4 — Calibration

PCRR: run stages in order, with the same MODE and BANK_EXTENSION. No P0.
Start SMOKE, then PILOT, then FULL. P5 opens Test **only in FULL after P4 freeze**.
SMOKE/PILOT P5 exports development results; it does not mark P5 PASS.
Read `README.md`, `reports/METHODOLOGY.md` and the generated acceptance report.
Data and checkpoints remain on Drive. Restart runtime after first dependency installation.


In [2]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FJCAI2027_PCRR').resolve()
assert (ROOT / 'data/raw/dataset_manifest.json').is_file(), 'Missing existing project/manifest'
assert (ROOT / 'run_all.py').is_file(), 'Wait for project files to sync to Drive'
probe = ROOT / '.colab_write_probe'
probe.write_text('ok'); probe.unlink()
print(ROOT)


Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR


In [3]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
print('After initial install, restart runtime, then rerun mount and remaining cells.')


After initial install, restart runtime, then rerun mount and remaining cells.


In [4]:
MODE = 'FULL'  # SMOKE -> PILOT -> FULL
BANK_EXTENSION = True  # Set True consistently for all FULL stages to include Bank
import os
os.environ['PCRR_ROOT'] = str(ROOT)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests/test_core.py')], cwd=ROOT, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '/content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/tests/test_core.py'], returncode=0)

In [6]:
import json
import subprocess
import sys

print(f"Requested mode: {MODE}; stage: P4; bank extension: {BANK_EXTENSION}")

# Discover the current experiment signature without modifying prior artifacts.
base_command = [sys.executable, str(ROOT / 'run_all.py'),
                '--root', str(ROOT), '--mode', MODE]
if BANK_EXTENSION:
    base_command.append('--bank')

def run_stage(stage):
    command = base_command + ['--stage', stage]
    result = subprocess.run(command, cwd=ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    experiment_lines = [line for line in result.stdout.splitlines()
                        if line.startswith('Experiment: ')]
    assert len(experiment_lines) == 1, f"Missing experiment ID during {stage}"
    experiment_id = experiment_lines[0].split(': ', 1)[1].strip()
    assert experiment_id.startswith(MODE.lower() + '-'), 'Experiment mode mismatch'
    return result, experiment_id

# Try P4 first: in an unchanged environment, this retains the fast path.
result, RUN_ID = run_stage('P4')
run_dir = ROOT / 'results' / RUN_ID
status_path = run_dir / 'status.json'

def read_status():
    if not status_path.is_file():
        return {}
    return json.loads(status_path.read_text(encoding='utf-8'))

if result.returncode != 0:
    # Only recover an unmet prerequisite; unrelated P4 errors must stay visible.
    if 'P4 blocked: P3 is not PASS' not in result.stdout:
        result.check_returncode()
    print('P3 is not PASS for the CURRENT experiment signature.')
    print('Running only missing prerequisites P1-P3; P5 will NOT run.')
    for stage in ('P1', 'P2', 'P3'):
        if read_status().get(stage) == 'PASS':
            print(f'{stage}: PASS; retaining verified artifacts')
            continue
        stage_result, stage_id = run_stage(stage)
        assert stage_id == RUN_ID, 'Experiment signature changed during recovery'
        stage_result.check_returncode()
        assert read_status().get(stage) == 'PASS', f'{stage} did not reach PASS'

    result, stage_id = run_stage('P4')
    assert stage_id == RUN_ID, 'Experiment signature changed during P4'
    result.check_returncode()

assert read_status().get('P4') == 'PASS', f'P4 did not reach PASS for {RUN_ID}'
print(f'P4 PASS for {RUN_ID}; P5 was not executed.')


Requested mode: FULL; stage: P4; bank extension: True
Experiment: full-62500e11e892b899
P4: starting
P4: PASS
Artifacts: /content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/results/full-62500e11e892b899

P4 PASS for full-62500e11e892b899; P5 was not executed.


In [7]:
import json
from IPython.display import Markdown, display

run_dir = ROOT / 'results' / RUN_ID
status = json.loads((run_dir/'status.json').read_text(encoding='utf-8'))
assert status['P4'] == 'PASS', f'P4 has not passed for {RUN_ID}'
frozen = json.loads((run_dir/'frozen_protocol.json').read_text(encoding='utf-8'))
assert frozen['experiment_id'] == RUN_ID
assert frozen['signature']['mode'] == MODE
report = ROOT / 'reports' / RUN_ID / 'P4_ACCEPTANCE.md'
display(Markdown(report.read_text(encoding='utf-8')))
print('Verified P4 PASS for:', RUN_ID)
print('Frozen protocol:', run_dir/'frozen_protocol.json')
print('P5 has not been executed by this notebook.')


# P4 Acceptance — full-62500e11e892b899

Status: **PASS**

## Work and evidence

Risk parameters selected on Risk-Val; clean Calibration thresholds frozen with code, library and artifact hashes.

## Artifacts

- results/full-62500e11e892b899/
- data/processed/full-62500e11e892b899/
- checkpoints/full-62500e11e892b899/

## Verification

`python -m pytest -q`

Stage command: `python run_all.py --mode FULL --stage P4`

Stage PASS refers to executed runtime assertions; unit-test evidence is recorded separately in reports/IMPLEMENTATION.md.

## Limits and next gate

SMOKE/PILOT are development-only, not publication evidence. P5 requires FULL and verified P4 artifacts. Scientific acceptance requires inspection of actual results.


Verified P4 PASS for: full-62500e11e892b899
Frozen protocol: /content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/results/full-62500e11e892b899/frozen_protocol.json
P5 has not been executed by this notebook.
